[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/38_causal_machine_learning/03_propensity_score_methods.ipynb)

# Bài 03 — Phương pháp điểm xu hướng (propensity score)

**🎯 Mục tiêu**
- **Điểm xu hướng** $e(x) = P(T = 1\mid X = x)$ và vì sao nó đủ để điều chỉnh confounding.
- Kiểm tra **overlap** (positivity) và **cân bằng** (standardized mean difference).
- **IPW** (Horvitz–Thompson, Hajek), **ghép cặp** (matching) cho ATT.
- **Doubly robust (AIPW)** với cross-fitting: đúng nếu **một trong hai** mô hình (kết quả hoặc xu hướng) đúng.
- Khi overlap bị vi phạm: trọng số bùng nổ, phương sai lớn — và cách xử lý.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/38_causal_machine_learning

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors

from ml_course.datasets import make_causal_marketing
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore")
data = make_causal_marketing(20000, seed=1)
FEATURES = ["age", "income_mvnd", "past_purchases"]
X, T, Y, tau = data[FEATURES].values, data.coupon.values, data.spend_kvnd.values, data.true_effect.values
ATE, ATT = tau.mean(), tau[T == 1].mean()
print(f"ATE thật = {ATE:.1f} | ATT thật = {ATT:.1f} | so sánh ngây thơ = {Y[T == 1].mean() - Y[T == 0].mean():.1f}")

## 1. Điểm xu hướng

Rosenbaum & Rubin (1983): nếu không có confounder chưa đo khi điều kiện theo $X$, thì cũng không có khi điều kiện theo **một số duy nhất** $e(X)$. Hai người có cùng xác suất nhận coupon (dù khác đặc điểm) có thể so sánh với nhau như trong thí nghiệm ngẫu nhiên. Ước lượng $e(x)$ bằng bất kỳ bộ phân loại nào — ở đây logistic regression.

In [ ]:
ps_model = LogisticRegression(max_iter=1000).fit(X, T)
e = ps_model.predict_proba(X)[:, 1]
print(f"điểm xu hướng: nhỏ nhất {e.min():.3f}, lớn nhất {e.max():.3f}")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(e[T == 1], bins=40, alpha=0.6, density=True, label="nhận coupon")
ax.hist(e[T == 0], bins=40, alpha=0.6, density=True, label="không nhận")
ax.set_xlabel("điểm xu hướng ê(x)")
ax.legend()
ax.set_title("overlap: hai phân phối phải chồng lên nhau")
plt.tight_layout()
plt.show()

Hai nhóm có điểm xu hướng **khác nhau** (người nhận coupon có $e$ cao hơn — đúng là có confounding) nhưng **chồng lên nhau** trên toàn dải và $e$ không quá gần 0 hay 1 → overlap đạt. Nếu một vùng chỉ có người được điều trị (hoặc chỉ có người không), dữ liệu **không** cho biết gì về phản thực tế ở vùng đó — không phương pháp nào cứu được (mục 6).

## 2. Inverse Probability Weighting (IPW)

Cân lại mỗi người theo nghịch đảo xác suất nhận điều trị **mà họ thực sự nhận**: người nhận coupon dù có $e$ thấp ("hiếm") được trọng số lớn. Dữ liệu sau khi cân giống một thí nghiệm ngẫu nhiên:

$$\widehat{\text{ATE}}_{\text{HT}} = \frac1n\sum_i\left[\frac{T_iY_i}{e(x_i)} - \frac{(1 - T_i)Y_i}{1 - e(x_i)}\right], \qquad \text{Hajek: chuẩn hoá trọng số mỗi nhóm về tổng 1}$$

In [ ]:
w1, w0 = T / e, (1 - T) / (1 - e)
ipw_ht = np.mean(T * Y / e - (1 - T) * Y / (1 - e))
ipw_hajek = (w1 * Y).sum() / w1.sum() - (w0 * Y).sum() / w0.sum()
print(f"IPW (Horvitz–Thompson) = {ipw_ht:.1f} | IPW (Hajek) = {ipw_hajek:.1f} | ATE thật = {ATE:.1f}")
print(f"trọng số lớn nhất = {max(w1.max(), w0.max()):.1f} | cỡ mẫu hiệu dụng nhóm điều trị = {w1.sum() ** 2 / (w1 ** 2).sum():,.0f} / {T.sum():,}")

### Kiểm tra cân bằng

Nếu trọng số đúng, **phân phối confounder sau khi cân phải giống nhau** giữa hai nhóm. Thước đo chuẩn: **SMD** (standardized mean difference) $= \frac{\bar x_1 - \bar x_0}{\sqrt{(s_1^2 + s_0^2)/2}}$; quy tắc thực hành: $|\text{SMD}| < 0.1$.

In [ ]:
def smd(x, t, w=None):
    w = np.ones_like(x, dtype=float) if w is None else w
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    return (m1 - m0) / np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2)


weights = T / e + (1 - T) / (1 - e)
balance = pd.DataFrame({"SMD trước khi cân": [smd(X[:, k], T) for k in range(3)],
                        "SMD sau IPW": [smd(X[:, k], T, weights) for k in range(3)]}, index=FEATURES)
print(balance.round(3))

Trước khi cân, `past_purchases` và `income` mất cân bằng mạnh (|SMD| > 0.1); sau IPW, mọi SMD ≈ 0. Cân bằng là **chẩn đoán** quan trọng nhất của mô hình xu hướng — một bộ phân loại có AUC cao **không** có nghĩa là trọng số tốt (AUC quá cao còn là dấu hiệu overlap kém).

## 3. Ghép cặp (matching) cho ATT

Với mỗi người nhận coupon, tìm người **không** nhận có điểm xu hướng gần nhất; so sánh kết quả. Ước lượng **ATT** (hiệu ứng trên những người thực sự được điều trị).

In [ ]:
nn = NearestNeighbors(n_neighbors=1).fit(e[T == 0].reshape(-1, 1))
dist, j = nn.kneighbors(e[T == 1].reshape(-1, 1))
att_matching = np.mean(Y[T == 1] - Y[T == 0][j[:, 0]])
matched_controls = X[T == 0][j[:, 0]]
print(f"ghép cặp 1–1 theo ê(x): ATT = {att_matching:.1f} | ATT thật = {ATT:.1f} | khoảng cách ê lớn nhất = {dist.max():.4f}")
print("SMD sau ghép cặp:", {f: round((X[T == 1][:, k].mean() - matched_controls[:, k].mean()) / X[:, k].std(), 3) for k, f in enumerate(FEATURES)})

Matching trực quan ("so sánh với người giống mình") nhưng bỏ phí dữ liệu, nhạy với cách ghép (có hoàn lại không, số láng giềng, ngưỡng khoảng cách) và có thiên lệch nhỏ khi cặp ghép chưa khớp hoàn toàn. Thực tế thường dùng ghép theo nhiều biến kèm caliper, hoặc dùng trọng số.

## 4. Doubly robust: AIPW

Kết hợp **mô hình kết quả** $\hat\mu_t(x)$ (bài 02) và **mô hình xu hướng** $\hat e(x)$:

$$\widehat{\text{ATE}}_{\text{AIPW}} = \frac1n\sum_i\left[\hat\mu_1(x_i) - \hat\mu_0(x_i) + \frac{T_i(Y_i - \hat\mu_1(x_i))}{\hat e(x_i)} - \frac{(1 - T_i)(Y_i - \hat\mu_0(x_i))}{1 - \hat e(x_i)}\right]$$

Số hạng hiệu chỉnh dùng trọng số xu hướng để sửa sai lệch còn lại của mô hình kết quả. **Doubly robust**: nhất quán nếu **một trong hai** mô hình đúng. Với mô hình ML linh hoạt, dùng **cross-fitting** (fit trên các fold khác, dự đoán trên fold còn lại — giống double machine learning, Chernozhukov et al., 2018) để tránh overfit làm chệch.

In [ ]:
def aipw(X, T, Y, outcome_model, propensity_model, n_folds=5, clip=0.01):
    mu1, mu0, e_hat = np.zeros(len(Y)), np.zeros(len(Y)), np.zeros(len(Y))
    for train_idx, test_idx in KFold(n_folds, shuffle=True, random_state=0).split(X):
        Xtr, Ttr, Ytr = X[train_idx], T[train_idx], Y[train_idx]
        mu1[test_idx] = outcome_model().fit(Xtr[Ttr == 1], Ytr[Ttr == 1]).predict(X[test_idx])
        mu0[test_idx] = outcome_model().fit(Xtr[Ttr == 0], Ytr[Ttr == 0]).predict(X[test_idx])
        e_hat[test_idx] = np.clip(propensity_model().fit(Xtr, Ttr).predict_proba(X[test_idx])[:, 1], clip, 1 - clip)
    scores = mu1 - mu0 + T * (Y - mu1) / e_hat - (1 - T) * (Y - mu0) / (1 - e_hat)
    return scores.mean(), scores.std() / np.sqrt(len(Y))


gbm_y, gbm_t = (lambda: HistGradientBoostingRegressor(random_state=0)), (lambda: HistGradientBoostingClassifier(random_state=0))
logit_t = lambda: LogisticRegression(max_iter=1000)


class MeanOnly:                                                         # mô hình kết quả "sai" (bỏ qua X)
    def fit(self, X, y):
        self.m = y.mean()
        return self

    def predict(self, X):
        return np.full(len(X), self.m)


class ConstantPropensity:                                               # mô hình xu hướng "sai" (bỏ qua X)
    def fit(self, X, t):
        self.p = t.mean()
        return self

    def predict_proba(self, X):
        return np.column_stack([np.full(len(X), 1 - self.p), np.full(len(X), self.p)])


rows = []
for name, om, pm in [("cả hai mô hình tốt (GBM)", gbm_y, gbm_t), ("mô hình kết quả SAI, xu hướng đúng", MeanOnly, logit_t),
                     ("mô hình kết quả đúng, xu hướng SAI", gbm_y, ConstantPropensity), ("cả hai SAI", MeanOnly, ConstantPropensity)]:
    est, se = aipw(X, T, Y, om, pm)
    rows.append({"AIPW với": name, "ước lượng": est, "KTC 95%": f"[{est - 1.96 * se:.1f}, {est + 1.96 * se:.1f}]"})
print(pd.DataFrame(rows).round(1).to_string(index=False))
print(f"ATE thật = {ATE:.1f}")

Đúng như tên gọi: chỉ cần **một** trong hai mô hình đúng là ước lượng gần ATE; cả hai sai thì quay về so sánh ngây thơ. AIPW cũng cho sai số chuẩn trực tiếp từ các "điểm số" từng người. Đây là nền của các thư viện hiện đại (EconML, DoubleML).

## 5. Tổng hợp

In [ ]:
summary = pd.DataFrame([
    ("so sánh ngây thơ", Y[T == 1].mean() - Y[T == 0].mean(), "ATE"),
    ("hồi quy OLS điều chỉnh", LinearRegression().fit(np.column_stack([T, X]), Y).coef_[0], "ATE"),
    ("IPW (Hajek)", ipw_hajek, "ATE"),
    ("AIPW (GBM, cross-fitting)", rows[0]["ước lượng"], "ATE"),
    ("ghép cặp theo ê(x)", att_matching, "ATT"),
], columns=["phương pháp", "ước lượng", "đại lượng"])
summary["giá trị thật"] = np.where(summary["đại lượng"] == "ATE", ATE, ATT)
print(summary.round(1).to_string(index=False))

## 6. Khi overlap bị vi phạm

Giả sử quy tắc phát coupon gần như **tất định**: khách có lịch sử mua ≥ 5 gần như chắc chắn nhận, khách ≤ 1 gần như không bao giờ nhận.

In [ ]:
rng = np.random.default_rng(3)
strict_p = np.clip(1 / (1 + np.exp(-(-6 + 2.2 * data.past_purchases.values))), 0.002, 0.998)
T_s = (rng.random(len(data)) < strict_p).astype(int)
Y_s = Y - tau * T + tau * T_s                                          # kết quả tiềm năng giữ nguyên, chỉ đổi ai được điều trị
e_s = LogisticRegression(max_iter=1000).fit(X, T_s).predict_proba(X)[:, 1]
w_s = T_s / e_s + (1 - T_s) / (1 - e_s)
print(f"ê(x): nhỏ nhất {e_s.min():.4f}, lớn nhất {e_s.max():.4f} | trọng số lớn nhất = {w_s.max():,.0f}")
for name, keep in [("không cắt", np.ones(len(e_s), bool)), ("bỏ ê ∉ [0.05, 0.95]", (e_s > 0.05) & (e_s < 0.95))]:
    est = np.mean(T_s[keep] * Y_s[keep] / e_s[keep] - (1 - T_s[keep]) * Y_s[keep] / (1 - e_s[keep]))
    print(f"IPW {name:20s}: {est:7.1f} | ATE thật của nhóm giữ lại = {tau[keep].mean():.1f} | giữ {keep.mean():.0%} dữ liệu")
boot = [np.mean((lambda i: T_s[i] * Y_s[i] / e_s[i] - (1 - T_s[i]) * Y_s[i] / (1 - e_s[i]))(rng.integers(0, len(Y), len(Y)))) for _ in range(200)]
print(f"độ lệch chuẩn bootstrap của IPW không cắt = {np.std(boot):.1f} (so với vài đơn vị khi overlap tốt)")

Vài người có $\hat e$ cực nhỏ mà vẫn nhận coupon nhận trọng số hàng trăm — ước lượng bị **một nhúm** người chi phối, phương sai bùng nổ. **Cắt** (trimming) vùng thiếu overlap ổn định ước lượng nhưng **đổi câu hỏi**: giờ ta ước lượng hiệu ứng cho nhóm còn lại (nơi dữ liệu có thông tin), không phải cho toàn bộ dân số. Không có phép màu: ở vùng không ai (hoặc ai cũng) được điều trị, dữ liệu không trả lời được câu hỏi phản thực tế — cần thí nghiệm hoặc giả định ngoại suy.

## 🧠 Kiểm tra nhanh

1. Điểm xu hướng là gì? Vì sao điều kiện theo nó là đủ?
2. Viết ước lượng IPW. Hajek khác Horvitz–Thompson thế nào?
3. Kiểm tra cân bằng bằng SMD ra sao? Vì sao AUC cao của mô hình xu hướng không phải mục tiêu?
4. Matching ước lượng ATE hay ATT? Nhược điểm?
5. "Doubly robust" nghĩa là gì? Vì sao cần cross-fitting?
6. Vi phạm overlap gây ra điều gì? Cắt trọng số đánh đổi điều gì?

## 📌 Tóm tắt

- $e(x)$ tóm tắt confounding; kiểm tra overlap và cân bằng (SMD < 0.1) trước khi tin ước lượng.
- IPW cân lại để giống RCT; matching cho ATT; AIPW kết hợp hai mô hình, đúng nếu một trong hai đúng, dùng cross-fitting với ML.
- Overlap kém → trọng số lớn, phương sai lớn; trimming đổi tập đối tượng được ước lượng.

## 📚 Tài liệu tham khảo

- Rosenbaum & Rubin (1983) — *The Central Role of the Propensity Score in Observational Studies for Causal Effects*
- Austin (2011) — *An Introduction to Propensity Score Methods for Reducing the Effects of Confounding in Observational Studies*
- Robins, Rotnitzky & Zhao (1994) — doubly robust estimation; Kang & Schafer (2007) — *Demystifying Double Robustness*
- Chernozhukov et al. (2018) — *Double/debiased machine learning for treatment and structural parameters*
- Crump et al. (2009) — *Dealing with limited overlap in estimation of average treatment effects*